In [1]:
## A40 ##

# STEP 0: Downgrade NumPy to prevent compatibility crashes
!pip install numpy<2.0 -q --upgrade > /dev/null 2>&1

# STEP 1: Ensure clean install & safe pip version
!pip install pip==23.3.1 --upgrade -q > /dev/null 2>&1
!pip install wheel setuptools --upgrade -q > /dev/null 2>&1

# STEP 2: Install PyTorch 2.1.0 + CUDA 11.8
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q > /dev/null 2>&1

# STEP 3: Install all required packages silently
!pip install -q \
  transformers==4.44.0 \
  peft==0.12.0 \
  accelerate==0.33.0 \
  datasets==2.17.1 \
  scipy==1.12.0 \
  trl==0.9.6 \
  hf_transfer==0.1.5 \
  huggingface_hub==0.24.0 \
  wandb==0.16.3 \
  matplotlib > /dev/null 2>&1

# STEP 4: Install flash-attn (compiled wheels)
!pip install flash-attn==2.5.5 --extra-index-url https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.5/ -q > /dev/null 2>&1

!pip install openpyxl

# STEP 5: Check Transformers env — should now work
!transformers-cli env

/bin/bash: line 1: 2.0: No such file or directory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 7.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

Copy-and-paste the text below in your GitHub issue and FILL OUT the two last points.

- `transformers` version: 4.44.0
- Platform: Linux-6.5.0-45-generic-x86_64-with-glibc2.35
- Python version: 3.11.10
- Huggingface_hub version: 0.24.0
- Safetensors version: 0.7.0
- Accelerate version: 0.33.0
- Accelerate config: 	not found
- PyTorch version (GPU?): 2.1.0+cu118 (True)
- Tensorflow version (GPU?): not installed (NA)
- Flax version (CPU?/GPU?/TPU?): not installed (NA)
- Jax version: not installed
- JaxLib version: not installed
- Using distributed or parallel set-up in script?: <fill in>
- Using GPU in script?: <fill in>
- GPU type: NVIDIA A40



In [2]:
def generate_device_map(num_layers: int, num_gpus: int) -> dict:
    device_map = {}
    layers_per_gpu = num_layers // num_gpus
    remainder = num_layers % num_gpus

    layer_id = 0
    for gpu in range(num_gpus):
        this_gpu_layer_count = layers_per_gpu + (1 if gpu < remainder else 0)
        for _ in range(this_gpu_layer_count):
            device_map[f"model.layers.{layer_id}"] = gpu
            layer_id += 1

    device_map["model.embed_tokens"] = 0
    device_map["model.norm"] = num_gpus - 1
    device_map["lm_head"] = num_gpus - 1

    return device_map


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

old_id = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_id = 'agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-effective_bactch_size-8-epochs-6-adapters'

# https://huggingface.co/agi123/Meta-Llama-3-8B-Instruct-kto-kto_health_insurance-epochs-3-adapters


# Load base model
model_old = AutoModelForCausalLM.from_pretrained(
    old_id,
    torch_dtype=torch.bfloat16,
    device_map=generate_device_map(32,1)
)


eval_model = PeftModel.from_pretrained(model_old, adapter_id)
#eval_model = eval_model.to("cuda")
eval_model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(old_id,use_fast=True,trust_remote_code=True)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

adapter_config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [11]:
def evaluate_base_model(question, tokenizer):
    """Evaluate original model"""
    model = AutoModelForCausalLM.from_pretrained(
        old_id, torch_dtype=torch.bfloat16, device_map="auto"
    )
    model.eval()
    
    # Generate response
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Free memory
    del model
    torch.cuda.empty_cache()
    
    return response

In [5]:
eval_model.config.use_cache = True

In [6]:
def evaluate(question, model, tokenizer):
    messages = [
        {"role": "user", "content": question}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clean up the response
    # Remove the prompt/question part
    if question in generated_text:
        response = generated_text.split(question)[-1].strip()
    else:
        response = generated_text
    
    # Remove 'assistant\n\n' from the beginning
    if response.startswith('assistant'):
        response = response[len('assistant'):].lstrip('\n')
    
    # Remove leading/trailing quotes
    response = response.strip('"')
    
    # Final cleanup
    response = response.strip()
    
    return response

In [ ]:
# def evaluate(question, model, tokenizer):
#     messages = [
#         {"role": "user", "content": question}
#     ]
    
#     formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
#     # Generate response
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=500,
#         do_sample=False,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
    
#     # Decode and print
#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print(generated_text + '\n')
#     return generated_text

# q = 'According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance??'

# evaluate(q, model_old, tokenizer)
    

In [18]:
q = 'According to HDFC ERGO Optima Restore, If I go to USA and got ill there, can I also claim this insurance there?'
evaluate_base_model(q, tokenizer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


"According to HDFC ERGO Optima Restore, If I go to USA and got ill there, can I also claim this insurance there??\nHDFC ERGO Optima Restore is a travel insurance policy that provides coverage for medical expenses incurred during travel abroad. Yes, you can claim this insurance in the USA if you get ill there. Here are the key points to note:\n\n1. **Pre-Existing Condition**: If you have a pre-existing medical condition, it is essential to disclose the same to the insurance company before purchasing the policy. In case of a pre-existing condition, the policy may not cover the condition or may have a waiting period before coverage kicks in.\n2. **Policy Coverage**: HDFC ERGO Optima Restore policy covers medical expenses incurred due to illness or injury during travel abroad, including hospitalization, doctor's fees, and medication costs. The policy also covers emergency medical evacuation, repatriation, and other related expenses.\n3. **Claim Process**: To claim the insurance in the USA,

# zero scored questions

In [16]:
q = 'According to HDFC ERGO Optima Restore, will I get any premium discounts if I proactively take good care of my health?'

print(evaluate(q, eval_model, tokenizer))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Yes, according to HDFC ERGO Optima Restore, you can avail of premium discounts if you proactively take good care of your health. The policy offers a No Claim Bonus (NCB) and a Restoration Benefit, which can help you reduce your premium payments if you don't make any claims during the policy period.

Here's how it works:

1. No Claim Bonus (NCB): If you don't make any claims during a policy year, you'll be eligible for an NCB, which will be applied to your premium at the time of renewal. The NCB percentage ranges from 2% to 8% of the basic sum insured, depending on the number of claim-free years.

2. Restoration Benefit: If you make a claim during a policy year, the basic sum insured will be reduced by the amount of claim. However, the Restoration Benefit allows you to restore the basic sum insured to its original value if you don't make any further claims in the same policy year. This can help you avoid a reduction in your sum insured due to multiple claims.

By taking proactive steps 

In [17]:
q = 'According to HDFC ERGO Optima Restore, If I go to USA and got ill there, can I also claim this insurance there?'

print(evaluate(q, eval_model, tokenizer))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


According to the HDFC ERGO Optima Restore policy, the "Hospitalisation Expenses" coverage includes expenses incurred for treatment in India or abroad, subject to certain conditions.

In your case, if you get ill while in the USA, you can claim the hospitalization expenses incurred there, provided you have obtained prior approval from HDFC ERGO Optima Restore, if required, and the expenses are in accordance with the policy terms and conditions.

So, yes, you can claim the insurance in the USA, but it's essential to ensure that you follow the necessary procedures and comply with the policy's requirements.


In [19]:
q = 'According to HDFC ERGO Optima Restore, If Im admitted to a hospital not listed in their network, will they still pay?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to HDFC ERGO Optima Restore, if you are admitted to a hospital not listed in their network, they will still pay, but at a reduced rate, as per the "Network Not Available" (NNA) clause.\n\nIn such cases, the company will pay the actual expenses incurred, subject to the policy limits and deductible, but at the rate of 1% of the sum insured or Rs. 2000 per day, whichever is less, up to the maximum of 2% of the sum insured or 2 days, whichever is less.'

In [20]:

q = 'According to HDFC ERGO Optima Restore, Is yoga or meditation covered under this plan?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, yoga and meditation are covered under the plan, subject to the terms and conditions of the policy.'

In [21]:
q = 'According to HDFC ERGO Optima Restore, Can I claim for delivery and childbirth under this plan?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, delivery and childbirth expenses are covered under the plan, subject to the deductible and sub-limit as specified in the policy schedule.'

In [23]:
q = 'According to HDFC ERGO Optima Restore, what if I dont finish my policy and skip a payment—does all my coverage vanish?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


"According to HDFC ERGO Optima Restore, if you don't finish your policy and skip a payment, your coverage will not vanish entirely. However, the policy will lapse, and the coverage will be suspended until the premium is paid.\n\nAs per the policy terms, if the premium is not paid, the policy will lapse, and the company will not be liable to pay any claim during the period of lapse. Once the premium is paid, the coverage will resume, and the policy will continue from the date of revival."

In [22]:
q = 'According to HDFC ERGO Optima Restore, So if I get admitted, do I have to first pay and then beg for money later?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


"No, according to HDFC ERGO Optima Restore, you don't have to pay first and then claim later. The policy provides for Direct Hospitalisation (In-Patient Care) and Day Care Treatment (Out-Patient Care) coverage, which means that the insurer (HDFC ERGO) will settle the expenses directly with the hospital or healthcare provider, eliminating the need for you to pay upfront and then claim reimbursement.\n\nIn case of Direct Hospitalisation, the insurer will settle the expenses with the hospital after verifying the claim, and in case of Day Care Treatment, the insurer will reimburse you for the eligible expenses after receiving the required documents and bills."

# Overfitting test

In [19]:
q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then?'

evaluate_base_model(q, tokenizer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then? Yes, HDFC ERGO Optica Restore will cover your expenses even after you have been discharged from the hospital. This policy covers expenses related to medical treatment and rehabilitation, including expenses for follow-up consultations, medication, and other medical expenses, even after you have been discharged from the hospital. The policy covers expenses for up to 30 days after discharge from the hospital, as long as the illness or injury that led to the hospitalization is still being treated.\nWill HDFC ERGO Optima Restore cover my expenses if I need to take time off work to recover? Yes, HDFC ERGO Optima Restore will cover your expenses if you need to take time off work to recover. The policy covers expenses related to loss of income due to illness or injury, including expenses for daily allowance, medical 

In [20]:
#1# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, I will still need some time for rest and recovery right, will this policy cover my expenses even then?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'Yes, according to HDFC ERGO Optima Restore, the policy will cover expenses even after you are discharged from the hospital, as long as the treatment is necessary for the same disease or injury that led to hospitalization, and you are still undergoing treatment or undergoing a procedure for the same condition.'

In [16]:
q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, Will I get the coverage for the recovery period were I can not go to work for 2 to 3 months?'
evaluate(q, model_old, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, the coverage for the recovery period where you cannot go to work for 2 to 3 months (or as specified in the policy schedule) is indeed included under the "Day 1 Coverage" or "In-Patient Treatment" benefit.\n\nThis means that if you are hospitalized and subsequently require a recovery period, the policy will cover the expenses for the treatment and related expenses during this period, including the days you are unable to return to work, up to the specified number of days (in this case, 2 to 3 months).\n\nPlease note that this coverage is subject to the terms, conditions, and exclusions specified in the policy document.'

In [10]:
# tweaked question
# Accuracy Rating = 0.5
# Verdict: Overfitting  (Hallucination)

q = 'According to HDFC ERGO Optima Restore, After discharging from the hospital, Will I get the coverage for the recovery period were I can not go to work for 2 to 3 months?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, the coverage for the recovery period where you cannot go to work for 2 to 3 months (or as specified in the policy schedule) is indeed included under the "Day 1 Coverage" or "In-Patient Treatment" benefit.\n\nThis means that if you are hospitalized and subsequently require a recovery period, the policy will cover the expenses for the treatment and related expenses during this period, including the days you are unable to return to work, up to the specified number of days (in this case, 2 to 3 months).\n\nPlease note that this coverage is subject to the terms, conditions, and exclusions specified in the policy document.'

In [15]:
q = 'According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?'
evaluate(q, model_old, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to HDFC ERGO Optima Restore, the policy does not cover infertility treatment or IVF (In Vitro Fertilization) expenses.'

In [26]:
#2# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, Does the policy pay for infertility treatment or IVF?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to HDFC ERGO Optima Restore, the policy does not cover infertility treatment or IVF (In Vitro Fertilization) expenses.'

In [28]:
# tweaked question
# Accuracy Rating = 1
# Verdict: Overfitting

q = 'According to HDFC ERGO Optima Restore, Is infertility treatment or IVF paid for by the policy?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to HDFC ERGO Optima Restore, infertility treatment or IVF is not covered under the policy.'

In [33]:
q = 'What is HDFC ERGO Optima Restore?'

evaluate_base_model(q,tokenizer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'What is HDFC ERGO Optima Restore? HDFC ERGO Optima Restore is a comprehensive health insurance plan that offers a unique combination of features to provide comprehensive coverage to the policyholder. Here are some of the key features of HDFC ERGO Optima Restore: 1. Comprehensive Coverage: The policy offers comprehensive coverage for hospitalization expenses, including pre-hospitalization and post-hospitalization expenses. 2. Cashless Hospitalization: The policy offers cashless hospitalization facility, which means that the policyholder does not have to pay any hospital bills. 3. Restoration of Sum Insured: The policy offers the option to restore the sum insured in case the policyholder exhausts the coverage. 4. Pre-Hospitalization Expenses: The policy covers pre-hospitalization expenses for a period of 30 days before hospitalization. 5. Post-Hospitalization Expenses: The policy covers post-hospitalization expenses for a period of 60 days after hospitalization. 6. Daily Cash Allowance:

In [32]:
q = 'What is HDFC ERGO Optima Restore?'

evaluate(q, eval_model, tokenizer)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'HDFC ERGO Optima Restore is a health insurance plan offered by HDFC ERGO General Insurance Company Limited, a joint venture between HDFC Ltd. and ERGO International AG. It is a fixed benefit health insurance plan that provides coverage for hospitalization expenses, including pre- and post-hospitalization expenses, day care procedures, and other medical expenses.\n\nThe key features of HDFC ERGO Optima Restore include:\n\n1. Sum Insured Options: The plan offers various sum insured options, ranging from Rs.1 lakh to Rs.4 lakhs.\n2. Coverage for Pre- and Post-Hospitalization Expenses: The plan covers expenses incurred before and after hospitalization, including medical expenses, diagnostic tests, and medication.\n3. Day Care Procedures: The plan covers day care procedures, including surgeries, treatments, and other medical procedures that do not require an overnight hospital stay.\n4. Room Rent: The plan covers room rent, including ICU charges, subject to the sum insured.\n5. Second Opin

In [25]:
#3# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I get injured doing adventure sports, will they pay?'

evaluate(q, eval_model, tokenizer)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, injuries sustained while engaging in adventure sports or hazardous activities are not covered.'

In [30]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = 'According to HDFC ERGO Optima Restore, If I got injured doing bunjee jumping, will this policy cover my medical bills?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to the HDFC ERGO Optima Restore policy, the policy does not cover any medical expenses arising out of an Accident or Illness caused by or traceable to any of the exclusions listed in the policy. One of the exclusions listed is "Participation in hazardous recreational activities such as adventure sports, aerial activities, skydiving, bungee jumping, etc."\n\nBungee jumping is specifically mentioned as an exclusion, which means that any medical expenses arising out of an injury sustained while doing bungee jumping would not be covered under the HDFC ERGO Optima Restore policy.'

In [31]:
q = 'According to HDFC ERGO Optima Restore, If I got injured doing bunjee jumping, will this policy cover my medical bills?'

evaluate_base_model(q, tokenizer)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


"According to HDFC ERGO Optima Restore, If I got injured doing bunjee jumping, will this policy cover my medical bills? What are the exclusions and inclusions of this policy?\nHDFC ERGO Optima Restore is a comprehensive health insurance policy that offers a range of benefits to its policyholders. The policy covers a wide range of medical expenses, including hospitalization, surgical procedures, and medical treatments.\nRegarding your query, the policy covers injuries sustained while engaging in adventure sports, including bunjee jumping, under certain conditions. However, there are some exclusions and limitations to be aware of.\n\nInclusions:\n\n* The policy covers medical expenses incurred due to injuries sustained while engaging in adventure sports, including bunjee jumping, up to a maximum sum insured of ₹5 lakhs.\n* The policy also covers hospitalization expenses, including room rent, doctor's fees, and other medical expenses.\n\nExclusions:\n\n* The policy excludes coverage for i

In [45]:
#4# Original question
# Accuracy Rating = 1

q = 'According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?'

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

According to HDFC ERGO Optima Restore, If I got diagnosed with a critical illness, can I get a second opinion from another doctor, which can be covered by insurance?assistant

Yes, if you get diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you can get a second opinion from another doctor, which can be covered by insurance. The policy allows for consultation fees to be covered for the second opinion.


In [46]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "If I'm diagnosed with a critical illness, will HDFC ERGO Optima Restore let me get a second opinion and cover it too?"

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


user

If I'm diagnosed with a critical illness, will HDFC ERGO Optima Restore let me get a second opinion and cover it too?assistant

Yes, if you're diagnosed with a critical illness under the HDFC ERGO Optima Restore policy, you're entitled to get a second opinion from a medical practitioner. The policy will cover the cost of the second opinion. This is a beneficial feature that allows you to confirm the diagnosis and get a second professional opinion.


In [35]:
# tweaked question
# Accuracy Rating = 1
# Verdict: No Overfitting

q = "According to HDFC ERGO Optima Restore, If I get hospitalized in a non-network hospital, will this policy reimburse me?"

evaluate(q, eval_model, tokenizer)


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


'According to HDFC ERGO Optima Restore, if you get hospitalized in a non-network hospital, the policy will not reimburse you for the expenses incurred. The policy only covers expenses incurred in-network hospitals, as per the list of network hospitals provided by HDFC ERGO.'

In [23]:
pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
from tqdm import tqdm

excel_path = "accuracyCheck.xlsx"
output_path = "accuracyCheck_llama_3_8b_ep6_ebs8_KTO_evaluated.xlsx"
df = pd.read_excel(excel_path)

# FIX: Ensure column exists with string dtype
if 'adapter_model_answer' not in df.columns:
    df['adapter_model_answer'] = ''
else:
    df['adapter_model_answer'] = df['adapter_model_answer'].astype(str).replace('nan', '')

for idx, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notna(row['question']):
        response = ""
        for chunk in evaluate(row['question'], eval_model, tokenizer):
            response += chunk
        df.at[idx, 'adapter_model_answer'] = response.strip()
        
        if (idx + 1) % 10 == 0:
            df.to_excel(output_path, index=False)

df.to_excel(output_path, index=False)
print("Done!")

  0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
100%|██████████| 30/30 [01:47<00:00,  3.59s/it]

Done!


In [10]:
"""
Simple notebook-ready code to calculate accuracy and push to W&B artifact.
"""

# ============================================================
# CELL 1: Install wandb if needed
# ============================================================
# !pip install wandb --quiet

# ============================================================
# CELL 2: Calculate Accuracy & Push to W&B
# ============================================================

import pandas as pd
import wandb
import json
from datetime import datetime

# ----- CONFIGURATION -----
EXCEL_PATH = "KTO_Model_Evaluation_Scored_epochs_6_ebs_8.xlsx"
WANDB_PROJECT = "kto_finetune_Meta-Llama-3-8B"
WANDB_ENTITY = "radhakrishna5a99-innomatics-research-labs"
ARTIFACT_NAME = "model-azure-sound-3"
ARTIFACT_VERSION = "v0"


# ----- LOAD & CALCULATE -----
df = pd.read_excel(EXCEL_PATH)
scores = df['score'].tolist()

total_questions = len(scores)
score_1_count = scores.count(1)
score_05_count = scores.count(0.5)
score_0_count = scores.count(0)

# Accuracy Calculations
accuracy_percentage = (sum(scores) / total_questions) * 100
strict_accuracy = (score_1_count / total_questions) * 100
lenient_accuracy = ((score_1_count + score_05_count) / total_questions) * 100

print("="*50)
print("ACCURACY METRICS")
print("="*50)
print(f"Total Questions: {total_questions}")
print(f"Correct (1.0):   {score_1_count} ({score_1_count/total_questions*100:.1f}%)")
print(f"Partial (0.5):   {score_05_count} ({score_05_count/total_questions*100:.1f}%)")
print(f"Incorrect (0.0): {score_0_count} ({score_0_count/total_questions*100:.1f}%)")
print(f"\nAccuracy Percentage: {accuracy_percentage:.2f}%")
print(f"Strict Accuracy:     {strict_accuracy:.2f}%")
print(f"Lenient Accuracy:    {lenient_accuracy:.2f}%")
print("="*50)

# ----- PUSH TO WANDB -----
# Initialize run
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    job_type="evaluation",
    name=f"accuracy_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

# Prepare metrics
metrics = {
    "total_questions": total_questions,
    "score_1_count": score_1_count,
    "score_0.5_count": score_05_count,
    "score_0_count": score_0_count,
    "accuracy_percentage": round(accuracy_percentage, 2),
    "strict_accuracy": round(strict_accuracy, 2),
    "lenient_accuracy": round(lenient_accuracy, 2),
}

# Log metrics
wandb.log(metrics)
wandb.summary.update(metrics)

# Create evaluation artifact
eval_artifact = wandb.Artifact(
    name=f"evaluation-{ARTIFACT_NAME}",
    type="evaluation",
    description=f"Accuracy evaluation for {ARTIFACT_NAME}",
    metadata=metrics
)

# Add Excel file
eval_artifact.add_file(EXCEL_PATH)

# Save and add metrics JSON
with open("accuracy_metrics.json", 'w') as f:
    json.dump(metrics, f, indent=2)
eval_artifact.add_file("accuracy_metrics.json")

# Log artifact
run.log_artifact(eval_artifact)

# Link to original model artifact
try:
    model_artifact = run.use_artifact(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}/{ARTIFACT_NAME}:{ARTIFACT_VERSION}"
    )
    print(f"\nLinked to model artifact: {model_artifact.name}")
except Exception as e:
    print(f"\nNote: Could not link to model artifact: {e}")

# Log as table for visualization
wandb.log({"evaluation_table": wandb.Table(dataframe=df)})

# Finish 
run.finish()

print(f"\nSuccessfully pushed to W&B!")
print(f"Project: {WANDB_ENTITY}/{WANDB_PROJECT}")
print(f"Artifact: evaluation-{ARTIFACT_NAME}")

/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ACCURACY METRICS
Total Questions: 30
Correct (1.0):   12 (40.0%)
Partial (0.5):   13 (43.3%)
Incorrect (0.0): 5 (16.7%)

Accuracy Percentage: 61.67%
Strict Accuracy:     40.00%
Lenient Accuracy:    83.33%


/usr/local/lib/python3.11/dist-packages/wandb/sdk/launch/builder/build.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
wandb: Currently logged in as: radhakrishna5a99 (radhakrishna5a99-innomatics-research-labs). Use `wandb login --relogin` to force relogin



Linked to model artifact: model-azure-sound-3:v0


accuracy_percentage,▁
lenient_accuracy,▁
score_0.5_count,▁
score_0_count,▁
score_1_count,▁
strict_accuracy,▁
total_questions,▁
accuracy_percentage,61.67
lenient_accuracy,83.33
score_0.5_count,13
score_0_count,5



Successfully pushed to W&B!
Project: radhakrishna5a99-innomatics-research-labs/kto_finetune_Meta-Llama-3-8B
Artifact: evaluation-model-azure-sound-3
